In [ ]:
import os
import pandas as pd
import re
from glob import glob
import xml.etree.ElementTree as ET
from collections import defaultdict
from datetime import datetime
from xml.dom import minidom
from typing import Dict, List, Tuple, Set
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
from bs4 import BeautifulSoup
from collections import Counter
import Levenshtein
from dataclasses import dataclass
import sys
sys.path.insert(0, "..")
from pipeline.data.preprocessing import preprocess_text
from pipeline.data.xml_utils import convert_to_xml
from pipeline.prompts import SYSTEM_PROMPT

In [ ]:
PSEUDO_LABEL_MODEL = "../SFT/Qwen3-1.7B-Base_SFT"  # path to a trained pseudo-label model
DATASET_NAME = "deepseek"

In [ ]:
if DATASET_NAME == "hermes":
    all_hermes_df = []
    for path in glob("../data/hermes/output_*_hermes.csv"):
        hermes = pd.read_csv(path)
        hermes.drop(index=hermes[hermes["features_to_include"].isna()].index, inplace=True)
        hermes.drop(index=hermes[hermes["default_response_1"].isna()].index, inplace=True)
        hermes['model'] = "Hermes"
        all_hermes_df.append(hermes)

    all_hermes_df = pd.concat(all_hermes_df)
    all_hermes_df.drop(columns=["index"], inplace=True)
    all_hermes_df.reset_index(inplace=True, drop=True)

    faulty_indices = [18765, 19717, 21618, 21362, 21524, 21507, 22274, 22506]
    all_hermes_df.drop(index=faulty_indices, inplace=True)
    all_hermes_df.reset_index(inplace=True, drop=True)
    final_df = all_hermes_df[["default_response_0", "default_response_1", "report_length", "features_to_include", "model"]]

In [ ]:
if DATASET_NAME == "haiku":
    all_haiku_df = []
    for path in glob("../data/haiku/output_*_haiku.csv"):
        haiku = pd.read_csv(path)
        haiku.rename(columns={"raw_report": "default_response_0", "anonymized_report": "default_response_1"}, inplace=True)
        haiku['model'] = "Haiku"
        all_haiku_df.append(haiku)
        
    all_haiku_df = pd.concat(all_haiku_df)
    all_haiku_df['report_length'] = all_haiku_df["criterion"].apply(lambda x: eval(x)["report_length"])
    all_haiku_df['features_to_include'] = all_haiku_df["criterion"].apply(lambda x: eval(x)["features_to_include"])
    all_haiku_df.drop(columns=["criterion"], inplace=True)
    all_haiku_df.reset_index(inplace=True, drop=True)
    final_df = all_haiku_df[["default_response_0", "default_response_1", "report_length", "features_to_include", "model"]]

In [ ]:
if DATASET_NAME == "deepseek":
    all_deepseek_df = []
    for path in glob("../data/deepseek/output_*_deepseek.csv"):
        deepseek = pd.read_csv(path)
        deepseek.rename(columns={"raw_report": "default_response_0", "anonymized_report": "default_response_1"}, inplace=True)
        deepseek['model'] = "DeepSeek"
        all_deepseek_df.append(deepseek)
        
    all_deepseek_df = pd.concat(all_deepseek_df)
    all_deepseek_df['report_length'] = all_deepseek_df["criterion"].apply(lambda x: eval(x)["report_length"])
    all_deepseek_df['features_to_include'] = all_deepseek_df["criterion"].apply(lambda x: eval(x)["features_to_include"])
    all_deepseek_df.drop(columns=["criterion"], inplace=True)
    all_deepseek_df.reset_index(inplace=True, drop=True)
    final_df = all_deepseek_df[["default_response_0", "default_response_1", "report_length", "features_to_include", "model"]]

In [ ]:
final_df["default_response_1"] = final_df["default_response_1"].str.replace("<anonymized_report>","").str.replace("</anonymized_report>","").str.strip().str.replace("<anonymized_report/>", "").str.replace("<raw_report>","").str.replace("</raw_report>","").str.strip().str.replace("<raw_report/>", "")
final_df["default_response_0"] = final_df["default_response_0"].str.replace("<anonymized_report>","").str.replace("</anonymized_report>","").str.strip().str.replace("<anonymized_report/>", "").str.replace("<raw_report>","").str.replace("</raw_report>","").str.strip().str.replace("<raw_report/>", "")

final_df.drop(index=final_df[final_df["default_response_0"].isna()].index, inplace=True)
final_df.drop(index=final_df[final_df["default_response_1"].isna()].index, inplace=True)
final_df.drop(index=final_df[final_df["default_response_0"] == ""].index, inplace=True)
final_df.drop(index=final_df[final_df["default_response_1"] == ""].index, inplace=True)

final_df.reset_index(inplace=True, drop=True)

## Preprocessing

In [ ]:
final_df = preprocess_text(final_df, input_col="default_response_0", output_col="default_response_1")

# Pseudolabeling

In [ ]:
def remove_anonymization_tags(text):
    # List of all tags to remove
    tags = ['name', 'age', 'date', 'place', 'address', 'phone', 'email', 'id', 'url', 'device']
    
    # Function to remove a specific tag
    def remove_tag(tag, text):
        pattern = rf'<{tag}(?:\s+[^>]*)?>(.*?)</{tag}>'
        return re.sub(pattern, r'\1', text)
    
    # Remove each tag
    for tag in tags:
        text = remove_tag(tag, text)
    
    return text

def handle_formatted_prop(text):
    tags = ['name', 'age', 'date', 'place', 'address', 'phone', 'email', 'id', 'url', 'device']
    
    def process_tag(tag, text):
        pattern = rf'<{tag}(\s+[^>]*?)?(formatted="([^"]*)")?(.*?)>(.*?)</{tag}>'
        
        def replace(match):
            prefix = match.group(1) or ''
            formatted_attr = match.group(2) or ''
            formatted_value = match.group(3)
            other_attrs = match.group(4) or ''
            content = match.group(5)
            
            # If formatted value exists, use it; otherwise, keep the original content
            new_content = formatted_value if formatted_value else content
            
            # Remove the formatted attribute if it exists
            attrs = (prefix + ' ' + other_attrs).strip()
            
            # Reconstruct the tag with the new content
            return f'<{tag}{" " + attrs if attrs else ""}>{new_content}</{tag}>'
        
        return re.sub(pattern, replace, text)
    
    # Process each tag
    for tag in tags:
        text = process_tag(tag, text)
    
    return text

def deanonymize_dataframe_v1(df, column_name, new_column_name):
    df[new_column_name] = df[column_name].apply(remove_anonymization_tags)
    df[column_name] = df[column_name].apply(handle_formatted_prop)
    return df

final_df = deanonymize_dataframe_v1(final_df, "default_response_1", "default_response_1_deanonymized")

In [ ]:
def deanonymize_dataframe_v2(df, column_name, xml_column_name, native_column_name):
    df[xml_column_name] = df[column_name].apply(convert_to_xml)
    df[native_column_name] = df[column_name].apply(remove_anonymization_tags)
    return df

final_df = deanonymize_dataframe_v2(final_df, "default_response_1", "xml_haiku", "native_text")

In [ ]:
final_df = final_df[["native_text", "xml_haiku", "report_length", "features_to_include"]]
final_df.head()

Run:
```bash
CUDA_VISIBLE_DEVICES="3" python -m vllm.entrypoints.openai.api_server --model ../SFT/Qwen3-1.7B-Base_SFT --port 9995 --tensor-parallel-size 1 
```


In [ ]:
from openai import OpenAI
client = OpenAI(api_key="EMPTY", base_url="http://localhost:9995/v1")
# generation task
def run_model(note):
    global client
    
    input_text = \
f"""<instructions>
{SYSTEM_PROMPT}
</instructions>

<note>
{note.strip()}
</note>

<phi_entities>
"""

    response = client.completions.create(
        model=PSEUDO_LABEL_MODEL,
        prompt=input_text,
        max_tokens=2048,
        temperature=0,
        seed=1995,
        stop=["</phi_entities>"]
    )
    
    response_text = "<phi_entities>\n" + response.choices[0].text
    response_text = response_text.split("</phi_entities>", 1)[0] + "</phi_entities>"

    return response_text

In [ ]:
final_df["xml_pseudolabel"] = ""

def process_row(i):
    return i, run_model(final_df.at[i, "native_text"])

with ThreadPoolExecutor(max_workers=100) as executor:
    futures = [executor.submit(process_row, i) for i in range(len(final_df))]
    for future in tqdm(as_completed(futures), total=len(final_df), desc="Processing rows"):
        i, result = future.result()
        final_df.at[i, "xml_pseudolabel"] = result

In [ ]:
text = """Bardia Khosravi 22yo fue al Hospital San Rafael el 22 de mayo de 2024 para ser atendida por el Dr. Gómez debido a su lesión cutánea."""
result = run_model(text)
print(result)

In [ ]:
exact_matches_indices = []
for i in range(len(final_df)):
    # check if the xml_pseudolabel and xml_haiku are the same
    if final_df.at[i, "xml_pseudolabel"] == final_df.at[i, "xml_haiku"]:
        exact_matches_indices.append(i)
print(f"Exact matches count: {len(exact_matches_indices)}")
print(f"Non matches count: {len(final_df) - len(exact_matches_indices)}")

In [ ]:
final_df[final_df["xml_pseudolabel"].str.startswith("<phi_entities>\n<names/>")]

In [ ]:
def clean_xml(xml_string):
    """Clean XML string and ensure proper structure with parent tags."""
    # Extract content between phi_entities tags
    start_tag = "<phi_entities>"
    end_tag = "</phi_entities>"
    
    start_idx = xml_string.find(start_tag) + len(start_tag)
    end_idx = xml_string.find(end_tag)
    content = xml_string[start_idx:end_idx].strip()
    
    if not content:
        return create_empty_xml_with_parents()
    
    return f"<phi_entities>\n{content}\n</phi_entities>"

def create_empty_xml_with_parents():
    """Create empty XML structure with all parent tags."""
    parent_tags = ['names', 'ages', 'dates', 'places', 'addresses', 'phones', 'emails', 'ids', 'urls', 'devices']
    content = "\n".join([f"<{tag}/>" for tag in parent_tags])
    return f"<phi_entities>\n{content}\n</phi_entities>"

@dataclass(frozen=True)
class EntityMention:
    text: str
    start: int
    end: int
    tag: str
    attributes: tuple

def extract_mentions(text: str, xml: str) -> List[EntityMention]:
    """Extract all entity mentions with their positions from text using XML annotations with parent tag support."""
    mentions = []
    
    try:
        soup = BeautifulSoup(xml, 'xml').find('phi_entities')
        if not soup:
            return mentions
        
        parent_tags = ['names', 'ages', 'dates', 'places', 'addresses', 'phones', 'emails', 'ids', 'urls', 'devices']
        
        for parent_tag in parent_tags:
            parent_element = soup.find(parent_tag)
            if parent_element:
                for tag in parent_element.find_all(recursive=False):
                    mentions.extend(_extract_mentions_from_tag(tag, text))
                
    except Exception as e:
        print(f"Error extracting mentions: {e}")
        return mentions
    
    # Resolve overlaps by keeping longer mentions
    mentions.sort(key=lambda x: (-len(x.text), x.start))
    final_mentions = []
    
    for mention in mentions:
        overlaps = False
        for existing in final_mentions:
            if not (mention.end <= existing.start or mention.start >= existing.end):
                overlaps = True
                break
        
        if not overlaps:
            final_mentions.append(mention)
            
    return final_mentions

def _extract_mentions_from_tag(tag, text: str) -> List[EntityMention]:
    """Helper function to extract mentions from a single tag."""
    mentions = []
    
    # Handle name tags specially
    if tag.name == 'name':
        variations = tag.find('variations')
        if variations:
            for val in variations.find_all('value'):
                content = val.text.strip() if val.text else ""
                if content:
                    for match in re.finditer(re.escape(content), text, re.IGNORECASE):
                        mentions.append(EntityMention(
                            text=content,
                            start=match.start(),
                            end=match.end(),
                            tag='name',
                            attributes=tuple(sorted(tag.attrs.items()))
                        ))
    # Handle date tags specially
    elif tag.name == 'date':
        original = tag.find('original')
        if original and original.text:
            content = original.text.strip()
            if content:
                for match in re.finditer(re.escape(content), text, re.IGNORECASE):
                    formatted_date = tag.find('formatted')
                    attrs = (('formatted', formatted_date.text.strip()),) if formatted_date and formatted_date.text else ()
                    mentions.append(EntityMention(
                        text=content,
                        start=match.start(),
                        end=match.end(),
                        tag='date',
                        attributes=attrs
                    ))
    # Handle other tags
    else:
        content = tag.text.strip() if tag.text else ""
        if content:
            for match in re.finditer(re.escape(content), text, re.IGNORECASE):
                mentions.append(EntityMention(
                    text=content,
                    start=match.start(),
                    end=match.end(),
                    tag=tag.name,
                    attributes=tuple(sorted(tag.attrs.items()))
                ))
    
    return mentions

def resolve_overlapping_mentions(mentions: List[EntityMention]) -> List[EntityMention]:
    """Resolve overlapping mentions using a priority-based approach."""
    mentions.sort(key=lambda x: (-len(x.text), x.start))
    
    tag_priority = {
        'name': 5,
        'date': 4,
        'id': 3,
        'phone': 3,
        'email': 3,
        'device': 2,
        'address': 2,
        'place': 2,
        'url': 1,
        'age': 1
    }
    
    final_mentions = []
    for mention in mentions:
        for existing in final_mentions:
            if not (mention.end <= existing.start or mention.start >= existing.end):
                if tag_priority.get(mention.tag, 0) > tag_priority.get(existing.tag, 0):
                    final_mentions.remove(existing)
                    final_mentions.append(mention)
                break
        else:
            final_mentions.append(mention)
    
    return final_mentions

def merge_xmls(text: str, gt_xml: str, llm_xml: str) -> str:
    """Merge ground truth and LLM-generated XML annotations with parent tag support."""
    
    gt_xml = clean_xml(gt_xml)
    llm_xml = clean_xml(llm_xml)
    
    # Extract mentions from both sources
    gt_mentions = extract_mentions(text, gt_xml)
    llm_mentions = extract_mentions(text, llm_xml)
    
    # Combine mentions
    all_mentions = gt_mentions + llm_mentions
    
    # Group similar mentions (using Levenshtein distance)
    mention_groups = defaultdict(list)
    for mention in all_mentions:
        added = False
        for key in mention_groups:
            if (mention.tag == key.tag and 
                Levenshtein.ratio(mention.text.lower(), key.text.lower()) > 0.8):
                mention_groups[key].append(mention)
                added = True
                break
        if not added:
            mention_groups[mention].append(mention)
    
    # Resolve overlapping mentions
    final_mentions = resolve_overlapping_mentions(list(mention_groups.keys()))
    
    # Generate merged XML with parent tag structure
    return generate_xml_with_parent_structure(final_mentions, mention_groups)

def generate_xml_with_parent_structure(final_mentions: List[EntityMention], mention_groups: Dict) -> str:
    """Generate XML with parent tag structure."""
    root = BeautifulSoup('<phi_entities></phi_entities>', 'xml')
    
    # Create parent elements
    parent_tags = ['names', 'ages', 'dates', 'places', 'addresses', 'phones', 'emails', 'ids', 'urls', 'devices']
    tags = ['name', 'age', 'date', 'place', 'address', 'phone', 'email', 'id', 'url', 'device']
    tag_to_parent = dict(zip(tags, parent_tags))
    
    parent_elements = {}
    for parent_tag in parent_tags:
        parent_elements[parent_tag] = root.new_tag(parent_tag)
        root.phi_entities.append(parent_elements[parent_tag])
    
    # Sort mentions by tag type and position
    final_mentions.sort(key=lambda x: (x.tag, x.start))
    
    # Group name mentions to merge variations
    name_mentions = defaultdict(set)
    non_name_mentions = []
    
    for mention in final_mentions:
        if mention.tag == 'name':
            name_mentions[mention.attributes].add(mention)
        else:
            non_name_mentions.append(mention)
    
    # Process name mentions first
    for attrs, mentions in name_mentions.items():
        tag = root.new_tag('name')
        for key, value in attrs:
            tag[key] = value
            
        variations = root.new_tag('variations')
        seen_values = set()
        
        for mention in mentions:
            similar_mentions = mention_groups[mention]
            for m in similar_mentions:
                if m.text.lower() not in seen_values:
                    value = root.new_tag('value')
                    value.string = m.text
                    variations.append(value)
                    seen_values.add(m.text.lower())
                    
        tag.append(variations)
        parent_elements['names'].append(tag)
    
    # Process non-name mentions
    for mention in non_name_mentions:
        tag = root.new_tag(mention.tag)
        parent_tag = tag_to_parent.get(mention.tag, 'names')  # Default fallback
        
        # Add attributes except formatted
        for key, value in mention.attributes:
            if key == "formatted":
                continue
            tag[key] = value
            
        if mention.tag == 'date':
            original = root.new_tag('original')
            original.string = mention.text
            formatted = root.new_tag('formatted')
            formatted.string = dict(mention.attributes).get('formatted', mention.text)
            tag.append(original)
            tag.append(formatted)
        else:
            tag.string = mention.text
            
        parent_elements[parent_tag].append(tag)
    
    return str(root)

def validate_merged_xml(text: str, merged_xml: str) -> bool:
    """Validate that all entities in the merged XML are present in the text."""
    try:
        soup = BeautifulSoup(merged_xml, 'xml')
        
        parent_tags = ['names', 'ages', 'dates', 'places', 'addresses', 'phones', 'emails', 'ids', 'urls', 'devices']
        
        for parent_tag in parent_tags:
            parent_element = soup.find(parent_tag)
            if parent_element:
                for tag in parent_element.find_all(recursive=False):
                    if not _validate_tag_in_text(tag, text):
                        return False

        return True
    except:
        return False

def _validate_tag_in_text(tag, text: str) -> bool:
    """Helper function to validate a single tag against text."""
    if tag.name == 'name':
        variations = tag.find('variations')
        if variations:
            found = False
            for val in variations.find_all('value'):
                if val.text and val.text.lower() in text.lower():
                    found = True
                    break
            return found
    elif tag.name == 'date':
        original = tag.find('original')
        if original and original.text:
            return original.text.lower() in text.lower()
    elif tag.text:
        return tag.text.lower() in text.lower()
    
    return True

def handle_xml_merging(df):
    """Process the entire dataframe with the improved merging logic and parent tag support."""
    merged_xmls = []
    for _, row in tqdm(df.iterrows(), total=len(df)):
        merged_xml = merge_xmls(
            row['native_text'],
            row['xml_haiku'],
            row['xml_pseudolabel']
        )
        
        # Validate the merged XML
        if not validate_merged_xml(row['native_text'], merged_xml):
            print(f"Warning: Invalid merged XML found for text: {row['native_text'][:100]}...")
            # Fall back to ground truth XML in case of validation failure
            merged_xml = clean_xml(row['xml_haiku'])
            
        merged_xmls.append(merged_xml)
        
    return merged_xmls 

In [ ]:
sample_orig = """<phi_entities>
<names\>

<ages>
<age>25</age>
</ages>

<dates>
<date>
<original>January 1, 2024</original>
<formatted>2024-01-01</formatted>
</date>
</dates>

<phones>
<phone>1234567890</phone>
</phones>

<emails>
<email>john.smith@example.com</email>
</emails>

<ids>
<id>1234567890</id>
</ids>

<urls/>

<devices>
<device>Dexom Pro rehab device</device>
<device>macbook</device>
</devices>
</phi_entities>"""

sample_psudo = """<phi_entities>
<names>
<name id='1' type='patient'>
<variations>
<value>John Smith</value>
</variations>
</name>
</names>

<ages>
<age>25</age>
</ages>

<dates>
<date>
<original>January 1, 2024</original>
<formatted>2024-01-01</formatted>
</date>
</dates>

<phones>
<phone>1234567890</phone>
</phones>

<emails>
<email>john25.smith@example.com</email>
</emails>

<ids>
<id>1234567890</id>
</ids>

<urls/>

<devices>
<device>Dexom Pro</device>
</devices>
</phi_entities>"""

text = "John Smith is 25 years old and was born on January 1, 2024. John is a patient. john25.smith@example.com"

# print(merge_xmls(text, sample_orig, sample_psudo))

In [ ]:
final_df["merged_xml"] = handle_xml_merging(final_df)

In [ ]:
def simple_text_anonymization(native_text, xml_string):
    # replace all the text in the xml with ***
    soup = BeautifulSoup(xml_string, 'xml').find('phi_entities')
    to_be_replaced = {}
    parent_tags = ['names', 'ages', 'dates', 'places', 'addresses', 'phones', 'emails', 'ids', 'urls', 'devices']
    for parent_tag in parent_tags:
        parent_element = soup.find(parent_tag)
        if parent_element:
            for tag in parent_element.find_all(recursive=False):
                if tag.name == 'name':
                    variation_values = [val.text for val in tag.find('variations').find_all('value')]          
                    for variation in variation_values:
                        to_be_replaced[variation] = "***Name***"
                if tag.name == 'date':
                    original_date = tag.find('original')
                    to_be_replaced[original_date.text] = "***Date***"
                else:
                    to_be_replaced[tag.text] = f"***{tag.name}***"

    # sort the to_be_replaced by the length of the key longest to shortest
    to_be_replaced = sorted(to_be_replaced.items(), key=lambda x: len(x[0]), reverse=True)
    for key, value in to_be_replaced:
        native_text = native_text.replace(key, value)
    return native_text

def verify_name_variations(native_text, xml_string):
    soup = BeautifulSoup(xml_string, 'xml')
    native_text = simple_text_anonymization(native_text, xml_string)

    for name in soup.find_all('name'):
        variation_values = [val.text for val in name.find('variations').find_all('value')]          
        for variation in variation_values:
            value_words = variation.split(" ")
            if len(value_words) > 1:
                for word in value_words:
                    if word in native_text and word not in variation_values:
                        # add the word to the variations
                        name.find('variations').append(BeautifulSoup(f'<value>{word}</value>', 'xml'))
                    if word.lower() in native_text:
                        name.find('variations').append(BeautifulSoup(f'<value>{word.lower()}</value>', 'xml'))
                        
    return str(soup)

for i in tqdm(range(len(final_df))):
    final_df.at[i, "merged_xml"] = verify_name_variations(final_df.at[i, "native_text"], final_df.at[i, "merged_xml"])

In [ ]:
sample_xml = """<phi_entities>
<name id='1' type='patient'>
<variations>
<value>John Smith</value>
</variations>
</name>
<age>25</age>
<date>
<original>2024-01-01</original>
<formatted>2024-01-01</formatted>
</date>
<phone>1234567890</phone>
<email>john25.smith@example.com</email>
<id>1234567890</id>
<url>https://www.example.com</url>
<device>Dexom Pro rehab device</device>
<device>macbook</device>
</phi_entities>"""

sample_text = "John Smith is 25 years old and was born on January 1, 2024. john is a patient. john25.smith@example.com"
# print(verify_name_variations(sample_text, sample_xml))

In [ ]:
def annotate_text_with_xml(native_text, xml_string):
    """
    Annotate native text with XML tags from the provided XML string.
    
    Args:
        native_text (str): The original text to be annotated
        xml_string (str): XML string containing the tags and their content
    
    Returns:
        str: Annotated text with XML tags
    """
    parent_tags = ['names', 'ages', 'dates', 'places', 'addresses', 'phones', 'emails', 'ids', 'urls', 'devices']
    # Parse XML
    soup = BeautifulSoup(xml_string, 'lxml-xml')
    # remove recursive tags unless they are name, or date
    for tag in soup.find_all():
        if tag.name not in ['name', 'date']:
            if tag.find_parent(tag.name):
                tag.decompose()
                
    # remove tags that are empty
    for tag in soup.find_all():
        if not tag.text.strip():
            tag.decompose()
    annotated_text = native_text
    
    # Create a list to store all replacements with their positions
    replacements = []
    
    # Process each tag in the XML
    for tag in soup.find_all():
        all_variations = []
        if tag.name not in ['phi_entities', 'variations', 'value', 'original', 'formatted'] + parent_tags:
            # Get tag content and attributes
            content = None
            formatted_content = None
            if tag.name == 'name':
                # For name tags, get all variations and sort them by length (largest to smallest)
                variations = tag.find('variations')
                for variation in variations.find_all('value'):
                    all_variations.append(variation.text.strip())
            elif tag.name == 'date':
                # For date tags, get the formatted date
                formatted_date = tag.find('formatted')
                original_date = tag.find('original')
                all_variations.append(original_date.text.strip())
                formatted_content = formatted_date.text.strip()
            else:
                all_variations.append(tag.text.strip())
            
            all_variations.sort(key=len, reverse=True)

            for content in all_variations:
                # Escape special regex characters in content
                escaped_content = re.escape(content)
                
                # Find all occurrences of the content in the text
                for match in re.finditer(escaped_content, native_text, re.IGNORECASE):
                    start_pos = match.start()
                    end_pos = match.end()
                    
                    # Create opening tag with attributes
                    opening_tag = f"<{tag.name}"
                    for attr, value in tag.attrs.items():
                        opening_tag += f' {attr}="{value}"'
                    opening_tag += ">"
                    
                    # Store the replacement with its position
                    if formatted_content is not None:
                        replacements.append((
                            start_pos,
                            end_pos,
                            f"{opening_tag}{formatted_content}</{tag.name}>"
                        ))
                    else:
                        replacements.append((
                            start_pos,
                            end_pos,
                            f"{opening_tag}{match.group()}</{tag.name}>"
                        ))
    # Sort replacements by start position in reverse order
    # (to avoid position shifts when making replacements)
    # and remove overlapping replacements
    non_overlapping_replacements = []
    
    # Sort replacements by start position in reverse order
    for start_pos, end_pos, replacement in sorted(replacements, key=lambda x: (x[1] - x[0]), reverse=True):
        if len(non_overlapping_replacements) == 0:
            non_overlapping_replacements.append((start_pos, end_pos, replacement))
            continue
        for i in range(len(non_overlapping_replacements)):
            if non_overlapping_replacements[i][0] <= start_pos and non_overlapping_replacements[i][1] >= end_pos: 
                # if the proposed replacement is inside a longer approved replacement
                break
        else:
            non_overlapping_replacements.append((start_pos, end_pos, replacement))
    
    # Assign the filtered non-overlapping replacements back to replacements
    replacements = non_overlapping_replacements
    
    # sort replacements by start position in reverse order
    replacements.sort(key=lambda x: x[0], reverse=True)
    
    # Apply replacements
    for start_pos, end_pos, replacement in replacements:
        annotated_text = (
            annotated_text[:start_pos] +
            replacement +
            annotated_text[end_pos:]
        )
    
    return annotated_text

annotated_texts = []
for _, row in tqdm(final_df.iterrows(), total=final_df.shape[0]):
    annotated_texts.append(annotate_text_with_xml(row["native_text"], row["merged_xml"]))

final_df["annotated_text"] = annotated_texts

In [ ]:
sample_orig = """<phi_entities>
<names>
<name id='1' type='patient'>
<variations>
<value>John</value>
<value>John Smith</value>
</variations>
</name>
</names>
<ages>
<age>25</age>
</ages>
<dates>
<date>
<original>January 1, 2024</original>
<formatted>2024-01-01</formatted>
</date>
</dates>
<phones>
<phone>1234567890</phone>
</phones>
<emails>
<email>john25.smith@example.com</email>
</emails>
<ids>
<id>1234567890</id>
</ids>
<urls>
<url>https://www.example.com</url>
<url>https://www.example.com</url>
</urls>
<ids>
<id>1234567890</id>
</ids>
<devices>
<device>Dexom Pro rehab device</device>
<device>macbook</device>
</devices>
</phi_entities>"""

text = "John Smith is 25 years old and was born on January 1, 2024. john is a patient. john25.smith@example.com"
# print(annotate_text_with_xml(text, sample_orig))

In [ ]:
final_df = final_df[["native_text", "annotated_text", "report_length", "features_to_include"]]
final_df.rename(columns={"native_text": "default_response_0", "annotated_text": "default_response_1"}, inplace=True)

In [ ]:
final_df

In [ ]:
if DATASET_NAME == "hermes":
    final_df.to_csv("../data/hermes/hermes_pseudolabeled.csv", index=False)
elif DATASET_NAME == "haiku":
    final_df.to_csv("../data/haiku/haiku_pseudolabeled.csv", index=False)
elif DATASET_NAME == "deepseek":
    final_df.to_csv("../data/deepseek/deepseek_pseudolabeled.csv", index=False)